# BronzeWork Incremental
Incremental Bronze Ingestion Logic with re-run safe watermark logic

## STEP 1 --- Imports and Setup

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog retailops")

In [0]:
spark.sql("create schema if not exists bronze_schema")

##STEP 2 -- Bronze Control Table

This table stores the watermark for each source table.
It helps the pipeline to remember:
1. the latest timestamp already processed
2. The latest primary key processed at that timestamp
3. How many rows written in the latest run

this is what makes bronze load incremental and rerun safe

In [0]:
spark.sql("""
          create table if not exists retailops.bronze_schema.ingestion_control(
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
          )
          using delta
          """
)

##STEP 3: Source Table Config
This cell defines which source tables will be loaded into bronze and which columns should be used as 
1. Primary Key
2. Timestamp/watermark col

it also creates a unique bronze_run_id for current pipeline run

In [0]:
tables_config = {
  "orders" : {"pk_col" : "order_id", "ts_col" : "updated_at"},
  "products" : {"pk_col" : "product_id","ts_col" : "updated_at"},
  "payments" : {"pk_col" : "payment_id","ts_col" : "processed_at"}
}
bronze_run_id = str(uuid.uuid4())
print("current bronze run id:",bronze_run_id)


##Step 4 -- Helper Functions
This cell contains reusable functions:

1. get_last_successful_watermark() reads the last processed watermark from control table

2. upsert_bronze_control() updates the control table after a successful bronze load

these functions keep the main load logic cleaner and easier to understand

In [0]:
def get_last_successful_watermark(table_name:str):
  ctrl = (
    spark.table("retailops.bronze_schema.ingestion_control")
    .filter(
      (F.col("layer")=="bronze") & (F.col("table_name")=="table_name")&
      (F.col("run_status")=="success")
    ).orderBy(F.col("updated_at").desc())
    .limit(1)
  )
  rows = ctrl.collect()

  if not rows:
    return None, None
  return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):
    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.utcnow()
        )],
            schema="""
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
            """
    )
    dt = DeltaTable.forName(spark,"retailops.bronze_schema.ingestion_control")
    (dt.alias("t")
     .merge(control_df.alias("s"),"t.table_name = s.table_name and t.layer=s.layer")
     .whenMatchedUpdate(set={
         "ts_col":"s.ts_col",
         "pk_col":"s.pk_col",
         "last_successful_ts":"s.last_successful_ts",
         "last_successful_pk":"s.last_successful_pk",
         "last_run_id":"s.last_run_id",
         "rows_written":"s.rows_written",
         "run_status":"s.run_status",
         "updated_at":"s.updated_at"
     })
     .whenNotMatchedInsertAll()
     .execute()
    )

#STEP 5 -- Bronze Incremental Load Loop
This is the main bronze logic.
for each table the notebook:
1. reads the last watermark
2. reads the source sql table
3. filters only new/changed rows
4. adds bronze audit columns
5. appends the rows into bronze delta table
6. updates the control table

this is the core incremental loading logic

In [0]:
for table_name, cfg in tables_config.items():
  pk_col = cfg["pk_col"]
  ts_col = cfg["ts_col"]
  source_table = f"`sqlserver`.dbo.{table_name}"
  target_table = f"`retailops`.bronze_schema.{table_name}_raw"
  last_successful_ts,last_successful_pk = get_last_successful_watermark(table_name)
  print(f"\n===== Processing {table_name} ===")
  print(f"Last successful ts: {last_successful_ts}")
  print(f"Last successful pk: {last_successful_pk}")
  print(f"source table: {source_table}")
  
  source_df=spark.read.table(source_table).withColumn(ts_col,F.col(ts_col).cast("timestamp"))
  if last_successful_ts is None:
    rows_to_load = source_df
  else:
    rows_to_load = source_df.filter(
      (F.col(ts_col) > F.lit(last_successful_ts)) |
      (
        (F.col(ts_col) == F.lit(last_successful_ts)) &
        (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
      )
    )
  
  rows_to_load = (
    rows_to_load
    .withColumn("bronze_ingested_at",F.current_timestamp())
    .withColumn("bronze_run_id",F.lit(bronze_run_id))
    .withColumn("bronze_source_table",F.lit(source_table))
  )
  rows_count = rows_to_load.count()
  print(f"{table_name} rows_to_load = {rows_count}")
  if rows_count == 0:
    print(f"No new rows for {table_name}")
    upsert_bronze_control(
      table_name,
      ts_col,
      pk_col,
      last_successful_ts,
      last_successful_pk,
      rows_count,
      bronze_run_id
    )
    continue

  rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

  max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
  max_pk = (
    rows_to_load
    .filter(F.col(ts_col) == F.lit(max_ts))
    .agg(F.max(pk_col).cast("long").alias("max_pk"))
    .collect()[0]["max_pk"]
  )
  upsert_bronze_control(table_name,ts_col,pk_col,max_ts,max_pk,rows_count,bronze_run_id)
  print(f"wrote {rows_count} rows to {target_table}")

#Step 6 -- QUICK Validation

In [0]:
print("Orders Bronze Count:", spark.sql("select count(*) from retailops.bronze_schema.orders_raw").collect()[0][0])
print("Products Bronze Count:", spark.sql("select count(*)from retailops.bronze_schema.products_raw").collect()[0][0])
print("Payments Bronze Count:", spark.sql("select count(*)from retailops.bronze_schema.payments_raw").collect()[0][0])
display(spark.sql("select * from retailops.bronze_schema.ingestion_control").orderBy("table_name"))